Imports

In [17]:
import re
import unicodedata
import numpy as np
import pandas as pd

## Loading Data

In [18]:
data = pd.read_csv('Data/GSIB_clean.csv')
print('Raw shape:', data.shape)
display(data.head(5))

Raw shape: (24111, 4)


,stock,headline,date,date_only
0,JPM,How The Banco Del Bajío (BMV:BBAJIO O) Valuati...,2026-04-09 21:11:04+00:00,2026-04-09
1,JPM,JPMorgan Chase &amp; Co. (JPM) – Among the 13 ...,2026-04-09 21:05:10+00:00,2026-04-09
2,JPM,Analyst Upgrades This Bank As Banking Stocks E...,2026-04-09 19:05:16+00:00,2026-04-09
3,JPM,Merrill Recruits Advisors Who Oversaw $2.8 Bil...,2026-04-09 18:55:00+00:00,2026-04-09
4,JPM,"Home Depot CFO Warns Demand Softening, Housing...",2026-04-09 16:04:22+00:00,2026-04-09


## Preprocessing configuration and cleaning

In [19]:
# Choose modeling text strategy: 'keep', 'mask', or 'remove'
ENTITY_MODE = 'mask'

# Full ticker universe for masking
bank_tickers = [
    'JPM', 'BAC', 'C', 'HSBC', 'IDCBY', 'GS', 'BNPQY', 'UBS', 'ACGBY', 'BACHY',
    'CICHY', 'BCS', 'MUFG', 'MS', 'WFC', 'BK', 'STT', 'DB', 'ING', 'SAN', 'RY',
    'TD', 'SCBFF', 'SCGLY', 'MFG', 'SMFG', 'BCMXY', 'GCRLY', 'BPCE'
 ]
BANK_TICKERS = sorted({ticker.upper().strip() for ticker in bank_tickers})

# Optional: map ticker symbols to common aliases to reduce entity-dominated topics
STOCK_ALIAS_MAP = {
    'JPM': [r'jpmorgan', r'j\.p\.\s*morgan'],
    'BAC': [r'bank\s+of\s+america', r'bofa'],
    'WFC': [r'wells\s+fargo'],
    'C': [r'citigroup', r'\bciti\b'],
    'GS': [r'goldman\s+sachs'],
    'MS': [r'morgan\s+stanley'],
    'HSBC': [r'\bhsbc\b'],
    'SAN': [r'santander', r'banco\s+santander'],
    'BCS': [r'barclays'],
    'DB': [r'deutsche\s+bank'],
    'UBS': [r'\bubs\b'],
    'RY': [r'royal\s+bank\s+of\s+canada', r'\brbc\b'],
    'TD': [r'toronto\-dominion', r'toronto\s+dominion'],
    'SCBFF': [r'standard\s+chartered', r'\bstanchart\b'],
    'SCGLY': [r'societe\s+generale', r'société\s+générale', r'\bsocgen\b'],
    'MFG': [r'mizuho\s+financial\s+group', r'\bmizuho\b'],
    'SMFG': [r'sumitomo\s+mitsui', r'\bsmbc\b'],
    'MUFG': [r'mitsubishi\s+ufj', r'\bufj\b'],
    'BK': [r'bank\s+of\s+new\s+york\s+mellon', r'\bbny\b'],
    'STT': [r'state\s+street'],
    'IDCBY': [r'industrial\s+and\s+commercial\s+bank\s+of\s+china', r'\bicbc\b'],
    'ACGBY': [r'agricultural\s+bank\s+of\s+china'],
    'CICHY': [r'china\s+construction\s+bank'],
    'BACHY': [r'bank\s+of\s+china'],
    'BCMXY': [r'bank\s+of\s+communications'],
    'ING': [r'\bing\b\s+groep'],
    'BNPQY': [r'bnp\s+paribas'],
    'GCRLY': [r'credit\s+agricole', r'crédit\s+agricole'],
    'BPCE': [r'\bbpce\b']
}

print('ENTITY_MODE:', ENTITY_MODE)
print('Ticker count configured:', len(BANK_TICKERS))

ENTITY_MODE: mask
Ticker count configured: 29


In [20]:
def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+\|\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+-\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def _mask_ticker_forms(text: str, ticker: str, replacement: str) -> str:
    out = text
    t = re.escape(ticker)
    out = re.sub(rf'\${t}\b', replacement, out)
    out = re.sub(rf'\b(?:NYSE|NASDAQ|AMEX|TSX|LSE|HKEX|TSE)\s*:\s*{t}\b', replacement, out, flags=re.IGNORECASE)
    out = re.sub(rf'\({t}\)', replacement, out)
    out = re.sub(rf'(?<!_)\b{t}\b(?!_)', replacement, out)
    return out

def handle_stock_entities(text: str, stock: str, mode: str = 'mask') -> str:
    cleaned = text
    replacement_ticker = ' __TICKER__ ' if mode == 'mask' else ' '
    replacement_company = ' __COMPANY__ ' if mode == 'mask' else ' '

    if mode in ('mask', 'remove'):
        # Generic cashtags and exchange-prefixed tickers
        cleaned = re.sub(r'\$[A-Z]{1,7}\b', replacement_ticker, cleaned)
        cleaned = re.sub(r'\b(?:NYSE|NASDAQ|AMEX|TSX|LSE|HKEX|TSE)\s*:\s*[A-Z0-9.]{1,10}\b', replacement_ticker, cleaned, flags=re.IGNORECASE)

        # Mask all configured bank tickers globally
        for ticker in BANK_TICKERS:
            cleaned = _mask_ticker_forms(cleaned, ticker, replacement_ticker)

        # Row stock ticker (extra safety)
        row_stock = '' if pd.isna(stock) else str(stock).upper().strip()
        if row_stock:
            cleaned = _mask_ticker_forms(cleaned, row_stock, replacement_ticker)

        # Mask aliases/names
        for ticker, alias_patterns in STOCK_ALIAS_MAP.items():
            for pattern in alias_patterns:
                cleaned = re.sub(pattern, replacement_company, cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def preprocess_headline(text: str, stock: str, mode: str = 'mask') -> str:
    cleaned = normalize_text(text)
    if mode in ('mask', 'remove', 'keep') and mode != 'keep':
        cleaned = handle_stock_entities(cleaned, stock, mode=mode)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

## Apply preprocessing

In [21]:
required_cols = ['stock', 'headline', 'date']
missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Short-text filtering thresholds
MIN_HEADLINE_CHARS = 20
MIN_HEADLINE_TOKENS = 4

df = data.copy()
df = df.dropna(subset=['stock', 'headline', 'date']).copy()
df['stock'] = df['stock'].astype(str).str.upper().str.strip()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['date_only'] = df['date'].dt.date

# Preserve raw headline and create variants
df['headline_raw'] = df['headline'].astype(str).str.strip()
df['headline_clean_keep'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='keep'), axis=1)
df['headline_clean_mask'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='mask'), axis=1)
df['headline_clean_remove'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='remove'), axis=1)

if ENTITY_MODE == 'keep':
    df['headline_model'] = df['headline_clean_keep']
elif ENTITY_MODE == 'remove':
    df['headline_model'] = df['headline_clean_remove']
else:
    df['headline_model'] = df['headline_clean_mask']

# Remove empty model headlines
df['headline_model'] = df['headline_model'].fillna('').str.strip()
df = df[df['headline_model'].str.len() > 0].copy()

# Remove very short headlines (chars + token count)
rows_before_short_filter = len(df)
headline_token_count = df['headline_model'].str.split().str.len()
short_mask = (df['headline_model'].str.len() < MIN_HEADLINE_CHARS) | (headline_token_count < MIN_HEADLINE_TOKENS)
removed_short = int(short_mask.sum())
df = df[~short_mask].copy()

# Remove duplicates after filtering
df = df.drop_duplicates(subset=['stock', 'date', 'headline_model']).sort_values('date').reset_index(drop=True)

print(f'Removed short headlines: {removed_short} (min chars={MIN_HEADLINE_CHARS}, min tokens={MIN_HEADLINE_TOKENS})')
print('Processed shape:', df.shape)
display(df[['stock', 'date', 'date_only', 'headline_raw', 'headline_model']].head(10))

Removed short headlines: 98 (min chars=20, min tokens=4)
Processed shape: (24013, 9)


,stock,date,date_only,headline_raw,headline_model
0,NDAQ,2025-04-09 00:00:19+00:00,2025-04-09,"Dave Portnoy Says What Many Are Thinking, 'If ...","Dave Portnoy Says What Many Are Thinking, 'If ..."
1,SAN,2025-04-09 09:17:00+00:00,2025-04-09,New Strong Buy Stocks for April 9th,New Strong Buy Stocks for April 9th
2,BNPQY,2025-04-09 09:19:38+00:00,2025-04-09,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...
3,DB,2025-04-09 09:19:38+00:00,2025-04-09,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...
4,DB,2025-04-09 12:32:54+00:00,2025-04-09,"Bessent Sees ‘Normal Deleveraging’ in Bonds, W...","Bessent Sees ‘Normal Deleveraging’ in Bonds, W..."
5,SAN,2025-04-09 12:43:00+00:00,2025-04-09,Best Value Stocks to Buy for April 9th,Best Value Stocks to Buy for April 9th
6,ICE,2025-04-09 12:55:00+00:00,2025-04-09,NYSE Content Advisory: Pre-Market update + NYS...,NYSE Content Advisory: Pre-Market update + NYS...
7,IVZ,2025-04-09 13:11:31+00:00,2025-04-09,Short sellers mint $159 billion profit in 6 da...,Short sellers mint $159 billion profit in 6 da...
8,TROW,2025-04-09 13:13:00+00:00,2025-04-09,Nuro secures $6 billion valuation in latest fu...,Nuro secures $6 billion valuation in latest fu...
9,SAN,2025-04-09 14:25:47+00:00,2025-04-09,Santander exploring options for $8B stake in P...,__COMPANY__ exploring options for $8B stake in...


## Quality checks

In [22]:
summary = {
    'rows': len(df),
    'unique_stocks': int(df['stock'].nunique()),
    'date_min': df['date'].min(),
    'date_max': df['date'].max(),
    'empty_model_headlines': int((df['headline_model'].str.len() == 0).sum())
}
print(summary)

print('\nTop stocks by article count:')
display(df['stock'].value_counts().head(15).to_frame('n_articles'))

df['raw_len'] = df['headline_raw'].str.len()
df['model_len'] = df['headline_model'].str.len()
print('\nHeadline length stats (raw vs model):')
display(df[['raw_len', 'model_len']].describe().T)

print('\nExamples of transformations:')
display(df[['stock', 'headline_raw', 'headline_clean_mask', 'headline_clean_remove', 'headline_model']].head(10))

{'rows': 24013, 'unique_stocks': 39, 'date_min': Timestamp('2025-04-09 00:00:19+0000', tz='UTC'), 'date_max': Timestamp('2026-04-09 21:34:38+0000', tz='UTC'), 'empty_model_headlines': 0}

Top stocks by article count:


,n_articles
stock,
BCS,996
CBOE,996
MS,994
UBS,991
C,989
WFC,985
BX,984
IVZ,983
GS,983



Headline length stats (raw vs model):


,count,mean,std,min,25%,50%,75%,max
raw_len,24013.0,71.834798,23.394358,19.0,58.0,69.0,84.0,504.0
model_len,24013.0,73.204556,23.677109,21.0,59.0,70.0,86.0,504.0



Examples of transformations:


,stock,headline_raw,headline_clean_mask,headline_clean_remove,headline_model
0,NDAQ,"Dave Portnoy Says What Many Are Thinking, 'If ...","Dave Portnoy Says What Many Are Thinking, 'If ...","Dave Portnoy Says What Many Are Thinking, 'If ...","Dave Portnoy Says What Many Are Thinking, 'If ..."
1,SAN,New Strong Buy Stocks for April 9th,New Strong Buy Stocks for April 9th,New Strong Buy Stocks for April 9th,New Strong Buy Stocks for April 9th
2,BNPQY,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...
3,DB,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...
4,DB,"Bessent Sees ‘Normal Deleveraging’ in Bonds, W...","Bessent Sees ‘Normal Deleveraging’ in Bonds, W...","Bessent Sees ‘Normal Deleveraging’ in Bonds, W...","Bessent Sees ‘Normal Deleveraging’ in Bonds, W..."
5,SAN,Best Value Stocks to Buy for April 9th,Best Value Stocks to Buy for April 9th,Best Value Stocks to Buy for April 9th,Best Value Stocks to Buy for April 9th
6,ICE,NYSE Content Advisory: Pre-Market update + NYS...,NYSE Content Advisory: Pre-Market update + NYS...,NYSE Content Advisory: Pre-Market update + NYS...,NYSE Content Advisory: Pre-Market update + NYS...
7,IVZ,Short sellers mint $159 billion profit in 6 da...,Short sellers mint $159 billion profit in 6 da...,Short sellers mint $159 billion profit in 6 da...,Short sellers mint $159 billion profit in 6 da...
8,TROW,Nuro secures $6 billion valuation in latest fu...,Nuro secures $6 billion valuation in latest fu...,Nuro secures $6 billion valuation in latest fu...,Nuro secures $6 billion valuation in latest fu...
9,SAN,Santander exploring options for $8B stake in P...,__COMPANY__ exploring options for $8B stake in...,exploring options for $8B stake in Polish bank...,__COMPANY__ exploring options for $8B stake in...


In [23]:
# Coverage check: any configured bank tickers still visible in headline_model?
remaining_counts = {}
for ticker in BANK_TICKERS:
    pattern = rf'(?<!_)\b{re.escape(ticker)}\b(?!_)'
    count = int(df['headline_model'].astype(str).str.contains(pattern, regex=True).sum())
    if count > 0:
        remaining_counts[ticker] = count

print('\nRemaining visible configured tickers in headline_model:', len(remaining_counts))
if remaining_counts:
    display(pd.DataFrame(sorted(remaining_counts.items(), key=lambda x: x[1], reverse=True), columns=['ticker', 'count']).head(20))
else:
    print('✅ All configured bank tickers are masked/removed from headline_model.')


Remaining visible configured tickers in headline_model: 0
✅ All configured bank tickers are masked/removed from headline_model.


## Save outputs for downstream pipelines

In [24]:
output_main = 'Data/GSIB_preprocessed.csv'
output_model = 'Data/GSIB_model_input.csv'

df.to_csv(output_main, index=False)

model_df = df[['stock', 'date', 'date_only', 'headline_model', 'headline_raw']].copy()
model_df = model_df.rename(columns={'headline_model': 'headline'})
model_df.to_csv(output_model, index=False)

print('Saved:')
print(f'  {output_main} -> {df.shape}')
print(f'  {output_model} -> {model_df.shape}')

Saved:
  Data/GSIB_preprocessed.csv -> (24013, 11)
  Data/GSIB_model_input.csv -> (24013, 5)
